# Pildigeneratsiooni rakenduste loomine (OpenAI)

Suurte keelemudelite (LLM) võimalused ei piirdu ainult tekstigeneratsiooniga. Samuti saate genereerida pilte tekstikirjelduste põhjal. Kujutised kui modaalne viis on kasulik MedTechis, arhitektuuris, turismis, mänguarenduses, turunduses ja mujal. Selles õppetükis töötame tänapäevaste **GPT Image** mudelitega OpenAI platvormil.

## Õpieesmärgid

Selle õppetüki lõpuks oskad:

- Selgitada, mis on pildigeneratsioon ja kus see on kasulik.
- Mõista `gpt-image` mudeliperekonda ja kuidas see erineb vanematest DALL·E mudelitest.
- Ehita pildigeneratsiooni rakendus ning redigeeri pilte.

## Mis on pildigeneratsioon?

Pildigeneratsioonimudelid loovad pildid tekstipõhise juhise põhjal. Moodne mudel, nagu `gpt-image`, õpib treeningu käigus seose teksti ja piltide vahel ning seejärel muudab juhises toodud müra iteratiivselt pildiks, mis vastab sinu tekstile.

Kaks tuntud pildimudelite perekonda on:

- **`gpt-image` (OpenAI)** — käesoleva õppetüki kasutatav uusim põlvkond. Toetab tekstist pildi genereerimist ja piltide redigeerimist (maskiga täitmine).
- **Midjourney** — populaarne kolmanda osapoole mudel oma teenuse ja Discordi-põhise töövooga.

> Vanemad OpenAI pildimudelid — **DALL·E 2** ja **DALL·E 3** — on vana põlvkonna mudelid. DALL·E 3 pole uutele juurutustele enam saadaval ning funktsioon `create_variation` eksisteeris ainult DALL·E 2-s. Uute rakenduste puhul kasuta `gpt-image` mudeleid.

> **Oluline:** `gpt-image` mudelid tagastavad genereeritud pildi **base64** formaadis (`b64_json`), mitte URL-ina. Sinu kood dekrüpteerib base64 stringi baitideks ja salvestab selle — pilt ei ole allalaadimiseks URL-ina saadaval.


## Oma esimese pildigeneratsiooni rakenduse loomine

Mida on vaja pildigeneratsiooni rakenduse loomiseks? Sul on vaja järgmisi raamatukogusid:

- **python-dotenv**, seda raamatukogu soovitatakse tungivalt kasutada, et hoida oma saladusi *.env* failis eemal koodist.
- **openai**, seda raamatukogu kasutad OpenAI API-ga suhtlemiseks.
- **pillow**, piltidega töötamiseks Pythonis.


1. Loo fail *.env* järgmise sisuga:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Kogu ülalmainitud teegid faili nimega *requirements.txt* järgmiselt:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Seejärel loo virtuaalne keskkond ja paigalda teegid:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windowsi puhul kasuta järgmist käsurea käsku, et luua ja aktiveerida oma virtuaalne keskkond:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Lisa järgmine kood faili nimega *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Loo OpenAI objekt (loeb OPENAI_API_KEY sinu .env failist)
    client = openai.OpenAI()


    try:
        # Loo kujutis, kasutades pildigeneratsiooni API-d
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Sisesta siia oma päringu tekst
            size='1024x1024',
            n=1
        )
        # Sea salvestatud kujutise kataloog
        image_dir = os.path.join(os.curdir, 'images')

        # Kui kataloogi ei eksisteeri, loo see
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Algata pildi tee (märgi, et failitüüp peaks olema png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image mudelid tagastavad pildi base64 formaadis (b64_json), mitte URL-ina
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Kuvatakse kujutis vaikimisi pildivaaturis
        image = Image.open(image_path)
        image.show()

    # püüdke vead kinni
    except openai.BadRequestError as err:
        print(err)

    ```

Selgitame seda koodi:

- Esiteks impordime vajalikud raamatukogud, sh OpenAI raamatukogu, dotenv raamatukogu, base64 mooduli ja Pillow raamatukogu.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Seejärel loome kliendi, mis loeb API võtit sinu ``.env`` failist.

    ```python
    # Loo OpenAI objekt
    client = openai.OpenAI()
    ```

- Järgmisena genereerime pildi:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Sisestage siia oma päringu tekst
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` mudelid tagastavad pildi **base64** stringina `data[0].b64_json` sees. Me dekodeerime selle baitideks ja kirjutame faili — URL-i allalaadimiseks ei ole.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Lõpuks avame pildi ja kasutame standardset pildi vaatajat selle kuvamiseks:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Rohkem üksikasju pildi genereerimisest

Vaatame `images.generate` parameetreid:

- **model** on kasutatav pildimudel (näiteks `gpt-image-1`).
- **prompt** on tekstiline kirjeldus, mida kasutatakse pildi genereerimiseks. Siin on see "Jänes hobusel, hoiab käes kommipulka, udusel aasadel, kus kasvavad nartsissid".
- **size** on genereeritud pildi suurus (näiteks `1024x1024`, `1536x1024`, `1024x1536` või `"auto"`).
- **n** on genereeritud piltide arv. Siin toodetakse üks pilt.

> Pildimudelid ei kasuta `temperature` parameetrit — see on teksti genereerimise juhtimiseks. Mitmekesisuse saamiseks kutsu API uuesti; mitmekesisuse vähendamiseks tee prompt spetsiifilisemaks.

## Täiendavad pildi genereerimise võimalused

Nagu nägime, saab Pythoniga paari koodireaga pildi genereerida. `gpt-image` mudelid suudavad ka olemasolevat pilti **redigeerida**. Kui anda pilt, valikuline **mask** (mis märgib ala, mida muuta), ja prompt, saad muuta osa pildist — näiteks lisada meie jänesele mütsi.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# muudatused tagastatakse ka base64-formaadis
edited_image = base64.b64decode(response.data[0].b64_json)
```

Algne pilt sisaldab ainult jänest; lõpp-pildil on jänesel müts peas.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
